In [32]:
# =============================================================
# CELLULE 1 — IMPORTS ET CONFIGURATION
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Seed pour la reproductibilité
SEED = 42
np.random.seed(SEED)

# Style des graphiques
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"]      = 12

# Options d'affichage pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",    100)
pd.set_option("display.precision",   2)

print("✓ Environnement configuré avec succès.")

✓ Environnement configuré avec succès.


In [33]:
# =============================================================
# CELLULE 2 — CHARGEMENT DU DATASET NETTOYÉ
# =============================================================

# On charge data_clean.csv — résultat du notebook 02
# C'est ce fichier qu'on va enrichir avec le feature engineering
df = pd.read_csv("../data/processed/data_clean.csv",
                 low_memory=False)

# Copie de travail
df_features = df.copy()

# --- Vérification ---
print("=" * 45)
print("DATASET CHARGÉ")
print("=" * 45)
print(f"  Lignes   : {df.shape[0]:,}")
print(f"  Colonnes : {df.shape[1]}")
print("=" * 45)

# Aperçu
display(df_features.head())

DATASET CHARGÉ
  Lignes   : 70,436
  Colonnes : 46


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary
0,Caucasian,Female,[50-60),2,1,1,8,Cardiology,77,6,33,0,0,0,401,997,560,8,Non mesuré,Non mesuré,Steady,No,No,No,No,No,No,Down,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,1
1,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,49,1,11,0,0,0,722,305,250,3,Non mesuré,Non mesuré,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,0
2,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,68,2,23,0,0,0,820,493,E880,9,Non mesuré,>7,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0
3,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,46,0,20,0,0,0,274,427,416,9,Non mesuré,>8,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Ch,Yes,0
4,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,49,0,5,0,0,0,590,220,250,3,Non mesuré,Non mesuré,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0


In [34]:
# =============================================================
# CELLULE 3 — ENCODAGE DE L'ÂGE
# =============================================================

# --- Milieu de tranche ---
age_mapping = {
    "[0-10)"  : 5,  "[10-20)" : 15,
    "[20-30)" : 25, "[30-40)" : 35,
    "[40-50)" : 45, "[50-60)" : 55,
    "[60-70)" : 65, "[70-80)" : 75,
    "[80-90)" : 85, "[90-100)": 95
}
df_features["age"] = df_features["age"].map(age_mapping)

# --- 3 groupes (basés sur le papier Strack et al. 2014) ---
# Figure 2 du papier montre 3 comportements distincts :
# [0-30)   → groupe 0 (jeunes, très peu de cas)
# [30-60)  → groupe 1 (adultes)
# [60-100) → groupe 2 (seniors, majoritaires et plus à risque)
def age_group(age):
    if age < 30:
        return 0
    elif age < 60:
        return 1
    else:
        return 2

df_features["age_group"] = df_features["age"].apply(age_group)

# --- Vérification ---
print("Encodage de l'âge :")
print(df_features[["age", "age_group"]].value_counts()
      .sort_index())

Encodage de l'âge :
age  age_group
5    0              152
15   0              529
25   0             1111
35   1             2666
45   1             6772
55   1            12325
65   2            15731
75   2            17873
85   2            11398
95   2             1879
Name: count, dtype: int64


In [35]:
# =============================================================
# CELLULE 4 — ENCODAGE DES VARIABLES BINAIRES
# =============================================================

# Ces variables n'ont que 2 valeurs possibles
# On les transforme en 0/1 — c'est ce qu'on appelle
# un encodage binaire

# --- Gender ---
# Female → 0, Male → 1
df_features["gender"] = df_features["gender"].map(
    {"Female": 0, "Male": 1}
)

# --- diabetesMed ---
# No → 0, Yes → 1
df_features["diabetesMed"] = df_features["diabetesMed"].map(
    {"No": 0, "Yes": 1}
)

# --- change ---
# No → 0 (pas de changement de médicament)
# Ch → 1 (changement de médicament)
df_features["change"] = df_features["change"].map(
    {"No": 0, "Ch": 1}
)

# --- Vérification ---
print("=" * 45)
print("ENCODAGE DES VARIABLES BINAIRES")
print("=" * 45)
for col in ["gender", "diabetesMed", "change"]:
    print(f"\n  {col} :")
    print(f"  {df_features[col].value_counts().to_dict()}")
    print(f"  Type : {df_features[col].dtype}")

ENCODAGE DES VARIABLES BINAIRES

  gender :
  {0: 37468, 1: 32968}
  Type : int64

  diabetesMed :
  {1: 53617, 0: 16819}
  Type : int64

  change :
  {0: 38763, 1: 31673}
  Type : int64


In [36]:
# =============================================================
# CELLULE 5 — ENCODAGE DES MÉDICAMENTS
# =============================================================

# Les colonnes de médicaments ont 4 valeurs possibles :
# "No"     → 0 : médicament non prescrit
# "Steady" → 1 : dose stable
# "Up"     → 2 : dose augmentée
# "Down"   → 3 : dose diminuée
# Cet ordre est logique : il reflète l'intensité du traitement

medicament_mapping = {
    "No"    : 0,
    "Steady": 1,
    "Up"    : 2,
    "Down"  : 3
}

# Liste des colonnes de médicaments
colonnes_medicaments = [
    "metformin", "repaglinide", "nateglinide",
    "chlorpropamide", "glimepiride", "acetohexamide",
    "glipizide", "glyburide", "tolbutamide",
    "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "insulin",
    "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone",
    "metformin-pioglitazone"
]

# Encodage
for col in colonnes_medicaments:
    if col in df_features.columns:
        df_features[col] = df_features[col].map(
            medicament_mapping)

# --- Vérification ---
print("=" * 45)
print("ENCODAGE DES MÉDICAMENTS")
print("=" * 45)
print(f"  Colonnes encodées : {len(colonnes_medicaments)}")
print(f"\n  Exemple — insulin :")
print(f"  {df_features['insulin'].value_counts().to_dict()}")
print(f"  Type : {df_features['insulin'].dtype}")

ENCODAGE DES MÉDICAMENTS
  Colonnes encodées : 23

  Exemple — insulin :
  {0: 34122, 1: 21897, 3: 7491, 2: 6926}
  Type : int64


In [37]:
# =============================================================
# CELLULE 6 — GROUPEMENT DES CODES ICD-9
# =============================================================

def grouper_icd9(code):
    """
    Regroupe les codes ICD-9 en catégories médicales
    selon Strack et al. (2014)
    """
    # Si la valeur est manquante ou inconnue
    if pd.isna(code) or code == "Unknown":
        return "Autre"

    # Codes ICD-9 commençant par E ou V
    # (causes externes et facteurs influençant la santé)
    if str(code).startswith("E") or str(code).startswith("V"):
        return "Autre"

    try:
        code_num = float(code)
    except ValueError:
        return "Autre"

    # Groupement selon le papier Strack et al. 2014
    if 390 <= code_num <= 459 or code_num == 785:
        return "Circulatoire"
    elif 460 <= code_num <= 519 or code_num == 786:
        return "Respiratoire"
    elif 520 <= code_num <= 579 or code_num == 787:
        return "Digestif"
    elif str(code).startswith("250"):
        return "Diabete"
    elif 800 <= code_num <= 999:
        return "Blessure"
    elif 710 <= code_num <= 739:
        return "Musculo_squelettique"
    elif 580 <= code_num <= 629 or code_num == 788:
        return "Genito_urinaire"
    elif 140 <= code_num <= 239:
        return "Neoplasmes"
    else:
        return "Autre"

# --- Application sur les 3 colonnes de diagnostic ---
for col in ["diag_1", "diag_2", "diag_3"]:
    df_features[col] = df_features[col].apply(grouper_icd9)
    print(f"\n{col} :")
    print(df_features[col].value_counts())


diag_1 :
diag_1
Circulatoire            21244
Autre                   12430
Respiratoire             9575
Digestif                 6564
Diabete                  5709
Blessure                 4817
Musculo_squelettique     3959
Genito_urinaire          3507
Neoplasmes               2631
Name: count, dtype: int64

diag_2 :
diag_2
Circulatoire            22143
Autre                   18541
Diabete                  9387
Respiratoire             7087
Genito_urinaire          5525
Digestif                 2886
Blessure                 1833
Neoplasmes               1712
Musculo_squelettique     1322
Name: count, dtype: int64

diag_3 :
diag_3
Autre                   21440
Circulatoire            20912
Diabete                 12310
Respiratoire             4775
Genito_urinaire          4249
Digestif                 2686
Blessure                 1416
Musculo_squelettique     1388
Neoplasmes               1260
Name: count, dtype: int64


In [38]:
# =============================================================
# CELLULE 7 — ONE-HOT ENCODING DES DIAGNOSTICS
# =============================================================

# pd.get_dummies crée une colonne binaire par catégorie
# prefix permet de garder le nom de la colonne originale
# drop_first supprime une catégorie par variable
# pour éviter la multicolinéarité

colonnes_diag = ["diag_1", "diag_2", "diag_3"]

df_features = pd.get_dummies(df_features,
                              columns=colonnes_diag,
                              prefix=colonnes_diag,
                              drop_first=True)

# --- Vérification ---
nouvelles_cols = [col for col in df_features.columns
                  if col.startswith("diag_")]

print(f"Nouvelles colonnes créées : {len(nouvelles_cols)}")
print("\nListe :")
for col in nouvelles_cols:
    print(f"  {col}")

print(f"\nDimensions du dataset : {df_features.shape}")

Nouvelles colonnes créées : 24

Liste :
  diag_1_Blessure
  diag_1_Circulatoire
  diag_1_Diabete
  diag_1_Digestif
  diag_1_Genito_urinaire
  diag_1_Musculo_squelettique
  diag_1_Neoplasmes
  diag_1_Respiratoire
  diag_2_Blessure
  diag_2_Circulatoire
  diag_2_Diabete
  diag_2_Digestif
  diag_2_Genito_urinaire
  diag_2_Musculo_squelettique
  diag_2_Neoplasmes
  diag_2_Respiratoire
  diag_3_Blessure
  diag_3_Circulatoire
  diag_3_Diabete
  diag_3_Digestif
  diag_3_Genito_urinaire
  diag_3_Musculo_squelettique
  diag_3_Neoplasmes
  diag_3_Respiratoire

Dimensions du dataset : (70436, 68)


In [39]:
# =============================================================
# CELLULE 8 — ENCODAGE race ET medical_specialty
# =============================================================

# --- Race → One-Hot Encoding ---
# Seulement 5 catégories → pas d'explosion de colonnes
df_features = pd.get_dummies(df_features,
                              columns=["race"],
                              prefix="race",
                              drop_first=True)

print("Race encodée ✓")
print([col for col in df_features.columns
       if col.startswith("race_")])

# --- Medical Specialty → Top 10 + "Autre" ---
# On garde les 10 spécialités les plus fréquentes
# Le reste devient "Autre" pour éviter trop de colonnes

top10_specialties = df_features["medical_specialty"] \
                    .value_counts().head(10).index.tolist()

print(f"\nTop 10 spécialités :")
for s in top10_specialties:
    print(f"  {s}")

# Remplacer les spécialités rares par "Autre"
df_features["medical_specialty"] = df_features[
    "medical_specialty"].apply(
    lambda x: x if x in top10_specialties else "Autre"
)

# One-Hot Encoding sur les 11 catégories résultantes
df_features = pd.get_dummies(df_features,
                              columns=["medical_specialty"],
                              prefix="specialty",
                              drop_first=True)

print(f"\nMedical specialty encodée ✓")
print(f"Dimensions actuelles : {df_features.shape}")

Race encodée ✓
['race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other']

Top 10 spécialités :
  Non mesuré
  InternalMedicine
  Family/GeneralPractice
  Emergency/Trauma
  Cardiology
  Surgery-General
  Orthopedics
  Orthopedics-Reconstructive
  Nephrology
  Radiologist

Medical specialty encodée ✓
Dimensions actuelles : (70436, 80)


In [40]:
# =============================================================
# CELLULE — CONVERSION BOOLÉENS EN ENTIERS
# =============================================================

# pd.get_dummies crée des colonnes True/False
# On les convertit en 1/0 pour XGBoost
# select_dtypes détecte toutes les colonnes booléennes

cols_bool = df_features.select_dtypes(include=["bool"]).columns

df_features[cols_bool] = df_features[cols_bool].astype(int)

print(f"✓ {len(cols_bool)} colonnes booléennes converties en 0/1")
print(f"  Dimensions : {df_features.shape}")

# Vérification
print("\nAperçu des colonnes converties :")
display(df_features[cols_bool[:5]].head())

✓ 38 colonnes booléennes converties en 0/1
  Dimensions : (70436, 80)

Aperçu des colonnes converties :


,diag_1_Blessure,diag_1_Circulatoire,diag_1_Diabete,diag_1_Digestif,diag_1_Genito_urinaire
0,0,1,0,0,0
1,0,0,0,0,0
2,1,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,1


In [41]:
# =============================================================
# CELLULE 9 — CRÉATION DE NOUVELLES VARIABLES
# =============================================================

# --- Variable 1 : Score de risque hospitalier ---
# Somme de toutes les visites médicales précédentes
# Plus ce score est élevé, plus le patient est fragile
df_features["score_risque_hospitalier"] = (
    df_features["number_inpatient"] +
    df_features["number_emergency"] +
    df_features["number_outpatient"]
)

# --- Variable 2 : Ratio médicaments / procédures ---
# +1 au dénominateur pour éviter la division par zéro
df_features["ratio_medicaments_procedures"] = (
    df_features["num_medications"] /
    (df_features["num_procedures"] + 1)
).round(2)

# --- Variable 3 : Patient complexe ---
# 1 si beaucoup de diagnostics ET beaucoup de médicaments
df_features["patient_complexe"] = (
    (df_features["number_diagnoses"] > 5) &
    (df_features["num_medications"] > 15)
).astype(int)

# --- Variable 4 : A1C mesuré ---
# 1 si le test A1C a été effectué
# Le papier montre que mesurer l'A1C réduit la réadmission
df_features["A1C_mesure"] = (
    df_features["A1Cresult"] != "Non mesuré"
).astype(int)



# --- Vérification ---
nouvelles_vars = [
    "score_risque_hospitalier",
    "ratio_medicaments_procedures",
    "patient_complexe",
    "A1C_mesure",
]

print("=" * 50)
print("NOUVELLES VARIABLES CRÉÉES")
print("=" * 50)
for var in nouvelles_vars:
    print(f"\n  {var} :")
    print(f"  {df_features[var].describe().to_dict()}")

print(f"\nDimensions actuelles : {df_features.shape}")

NOUVELLES VARIABLES CRÉÉES

  score_risque_hospitalier :
  {'count': 70436.0, 'mean': 0.7659151570219774, 'std': 1.6757805198788414, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 1.0, 'max': 68.0}

  ratio_medicaments_procedures :
  {'count': 70436.0, 'mean': 8.895036061105117, 'std': 5.604883721163976, 'min': 0.2, '25%': 4.5, '50%': 7.71, '75%': 12.0, 'max': 50.0}

  patient_complexe :
  {'count': 70436.0, 'mean': 0.37320404338690444, 'std': 0.4836590808640783, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 1.0, 'max': 1.0}

  A1C_mesure :
  {'count': 70436.0, 'mean': 0.17796297347947074, 'std': 0.3824842879528134, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 1.0}

Dimensions actuelles : (70436, 84)


In [42]:
# =============================================================
# CELLULE 10 — ENCODAGE A1Cresult ET max_glu_serum
# =============================================================

# --- A1Cresult ---
# ">8"        → 3 (taux très élevé, diabète mal contrôlé)
# ">7"        → 2 (taux élevé)
# "Normal"    → 1 (taux normal)
# "Non mesuré"→ 0 (pas de mesure effectuée)
A1C_mapping = {
    "Non mesuré": 0,
    "Normal"    : 1,
    ">7"        : 2,
    ">8"        : 3
}
df_features["A1Cresult"] = df_features["A1Cresult"].map(
    A1C_mapping)

print("A1Cresult encodé :")
print(df_features["A1Cresult"].value_counts().sort_index())

# --- max_glu_serum ---
# ">300"      → 3 (taux très élevé)
# ">200"      → 2 (taux élevé)
# "Normal"    → 1 (taux normal)
# "Non mesuré"→ 0 (pas de mesure effectuée)
glu_mapping = {
    "Non mesuré": 0,
    "Normal"    : 1,
    ">200"      : 2,
    ">300"      : 3
}
df_features["max_glu_serum"] = df_features[
    "max_glu_serum"].map(glu_mapping)

print("\nmax_glu_serum encodé :")
print(df_features["max_glu_serum"].value_counts().sort_index())

print(f"\nDimensions actuelles : {df_features.shape}")

A1Cresult encodé :
A1Cresult
0.0    57901
2.0     2852
3.0     5993
Name: count, dtype: int64

max_glu_serum encodé :
max_glu_serum
0.0    67046
2.0      949
3.0      701
Name: count, dtype: int64

Dimensions actuelles : (70436, 84)


In [43]:
# =============================================================
# CELLULE 11 — VÉRIFICATION FINALE
# =============================================================

print("=" * 55)
print("BILAN DU FEATURE ENGINEERING")
print("=" * 55)

# --- Dimensions ---
print(f"\n  Lignes   : {df_features.shape[0]:,}")
print(f"  Colonnes : {df_features.shape[1]}")

# --- Valeurs manquantes ---
manquantes = df_features.isnull().sum()
manquantes = manquantes[manquantes > 0]

print("\n" + "=" * 55)
print("VALEURS MANQUANTES")
print("=" * 55)
if len(manquantes) == 0:
    print("  ✓ Aucune valeur manquante !")
else:
    for col, val in manquantes.items():
        print(f"  {col:<40} {val}")

# --- Types des colonnes ---
print("\n" + "=" * 55)
print("TYPES DES COLONNES")
print("=" * 55)
types = df_features.dtypes.value_counts()
for dtype, count in types.items():
    print(f"  {str(dtype):<15} : {count} colonnes")

# --- Colonnes non numériques restantes ---
non_numeriques = df_features.select_dtypes(
    include=["object"]).columns.tolist()

print("\n" + "=" * 55)
print("COLONNES NON NUMÉRIQUES RESTANTES")
print("=" * 55)
if len(non_numeriques) == 0:
    print("  ✓ Toutes les colonnes sont numériques !")
else:
    for col in non_numeriques:
        print(f"  ⚠️  {col}")

# --- Aperçu final ---
print("\n" + "=" * 55)
print("APERÇU DU DATASET FINAL")
print("=" * 55)
display(df_features.head())

BILAN DU FEATURE ENGINEERING

  Lignes   : 70,436
  Colonnes : 84

VALEURS MANQUANTES
  max_glu_serum                            1740
  A1Cresult                                3690

TYPES DES COLONNES
  int64           : 81 colonnes
  float64         : 3 colonnes

COLONNES NON NUMÉRIQUES RESTANTES
  ✓ Toutes les colonnes sont numériques !

APERÇU DU DATASET FINAL


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary,age_group,diag_1_Blessure,diag_1_Circulatoire,diag_1_Diabete,diag_1_Digestif,diag_1_Genito_urinaire,diag_1_Musculo_squelettique,diag_1_Neoplasmes,diag_1_Respiratoire,diag_2_Blessure,diag_2_Circulatoire,diag_2_Diabete,diag_2_Digestif,diag_2_Genito_urinaire,diag_2_Musculo_squelettique,diag_2_Neoplasmes,diag_2_Respiratoire,diag_3_Blessure,diag_3_Circulatoire,diag_3_Diabete,diag_3_Digestif,diag_3_Genito_urinaire,diag_3_Musculo_squelettique,diag_3_Neoplasmes,diag_3_Respiratoire,race_Asian,race_Caucasian,race_Hispanic,race_Other,specialty_Cardiology,specialty_Emergency/Trauma,specialty_Family/GeneralPractice,specialty_InternalMedicine,specialty_Nephrology,specialty_Non mesuré,specialty_Orthopedics,specialty_Orthopedics-Reconstructive,specialty_Radiologist,specialty_Surgery-General,score_risque_hospitalier,ratio_medicaments_procedures,patient_complexe,A1C_mesure
0,0,55,2,1,1,8,77,6,33,0,0,0,8,0.0,0.0,1,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,1,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,4.71,1,0
1,0,55,3,1,1,2,49,1,11,0,0,0,3,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,5.50,0,0
2,0,85,1,3,7,4,68,2,23,0,0,0,9,0.0,2.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,7.67,1,1
3,0,85,1,1,7,3,46,0,20,0,0,0,9,0.0,3.0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,20.00,1,1
4,0,35,1,1,7,5,49,0,5,0,0,0,3,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,5.00,0,0


In [44]:
# =============================================================
# CELLULE 12 — SAUVEGARDE DU DATASET FINAL
# =============================================================

chemin = "../data/processed/data_features.csv"

df_features.to_csv(chemin, index=False)

print(f"✓ Dataset sauvegardé avec succès.")
print(f"  Chemin   : {chemin}")
print(f"  Lignes   : {df_features.shape[0]:,}")
print(f"  Colonnes : {df_features.shape[1]}")

✓ Dataset sauvegardé avec succès.
  Chemin   : ../data/processed/data_features.csv
  Lignes   : 70,436
  Colonnes : 84


## Conclusions du Feature Engineering

### 1. Encodages effectués
| Variable | Transformation |
|---|---|
| `age` | Milieu de tranche + age_group (3 catégories) |
| `gender` | Binaire (0/1) |
| `diabetesMed` | Binaire (0/1) |
| `change` | Binaire (0/1) |
| `A1Cresult` | Ordinal (0→3) |
| `max_glu_serum` | Ordinal (0→3) |
| Médicaments (23) | Ordinal (0→3) |
| `diag_1/2/3` | One-Hot (9 catégories ICD-9) |
| `race` | One-Hot (5 catégories) |
| `medical_specialty` | One-Hot (Top 10 + Autre) |

### 2. Nouvelles variables créées
| Variable | Description |
|---|---|
| `score_risque_hospitalier` | Total visites médicales précédentes |
| `ratio_medicaments_procedures` | Médicaments / procédures |
| `patient_complexe` | Cas complexe (diagnoses>5 ET meds>15) |
| `A1C_mesure` | HbA1c mesuré ou non |

### 3. Dataset final
- **70 436 lignes**
- **84 colonnes**
- **0 valeur manquante**
- **100% numérique** → prêt pour XGBoost

### 4. Source
Groupement ICD-9 basé sur Strack et al. (2014)

### 5. Prochaine étape
**Notebook 04** — Modélisation XGBoost